# Phase: Scrub
## Ziel: Bereinigung, NRW-Filter, saubere numerische Spalten 

In [1]:
import pandas as pd
from pathlib import Path

in_path = Path("../../data/processed/student_housing_de_obtain.csv")
df_de = pd.read_csv(in_path)
df_de.shape, df_de.head()


C:\Users\User\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


((248, 5),
          hochschulort wohnheime_2025_anzahl wohnheimplaetze_2025  \
 0               Aalen                     7                  531   
 1            Albstadt                     2                   82   
 2  Bad Mergentheim 4)                     2                   35   
 3            Biberach                     1                   64   
 4        Esslingen 5)                     5                  857   
 
   studierende_ws_2024_2025 studierende_je_wohnheimplatz_2025  
 0                     4117                                 8  
 1                     1392                                17  
 2                      550                                16  
 3                     2122                                33  
 4                     4807                                 6  )

In [2]:
df_de["hochschulort"] = (
    df_de["hochschulort"]
    .astype(str)
    .str.replace(r"\s*\d+\)\s*$", "", regex=True)   # entfernt " 4)" am Ende
    .str.strip()
)
df_de["hochschulort"].head(10)


0              Aalen
1           Albstadt
2    Bad Mergentheim
3           Biberach
4          Esslingen
5           Freiburg
6    Friedrichshafen
7         Furtwangen
8         Geislingen
9          Göppingen
Name: hochschulort, dtype: object

In [3]:
num_cols = [
    "wohnheime_2025_anzahl",
    "wohnheimplaetze_2025",
    "studierende_ws_2024_2025",
    "studierende_je_wohnheimplatz_2025"
]

for c in num_cols:
    df_de[c] = pd.to_numeric(df_de[c], errors="coerce")

df_de[num_cols].dtypes, df_de[num_cols].isna().sum()


(wohnheime_2025_anzahl                float64
 wohnheimplaetze_2025                 float64
 studierende_ws_2024_2025             float64
 studierende_je_wohnheimplatz_2025    float64
 dtype: object,
 wohnheime_2025_anzahl                48
 wohnheimplaetze_2025                 46
 studierende_ws_2024_2025             44
 studierende_je_wohnheimplatz_2025    46
 dtype: int64)

In [4]:
nrw_cities = [
    "Aachen", "Bielefeld", "Bochum", "Bonn", "Dortmund", "Duisburg",
    "Düsseldorf", "Essen", "Gelsenkirchen", "Hagen", "Köln",
    "Krefeld", "Mönchengladbach", "Mülheim an der Ruhr",
    "Münster", "Oberhausen", "Paderborn", "Siegen",
    "Wuppertal"
]


In [5]:
df_nrw = df_de[df_de["hochschulort"].isin(nrw_cities)].copy()
df_nrw.shape, df_nrw.head()


((17, 5),
     hochschulort  wohnheime_2025_anzahl  wohnheimplaetze_2025  \
 117       Aachen                   35.0                5460.0   
 118    Bielefeld                   25.0                3193.0   
 120       Bochum                   37.0                5594.0   
 121         Bonn                   35.0                4053.0   
 123     Dortmund                   13.0                2765.0   
 
      studierende_ws_2024_2025  studierende_je_wohnheimplatz_2025  
 117                   56493.0                               10.0  
 118                   31399.0                               10.0  
 120                   49371.0                                9.0  
 121                   34744.0                                9.0  
 123                   42769.0                               15.0  )

In [6]:
df_nrw["wohnheimplatzrelation"] = (
    df_nrw["studierende_ws_2024_2025"] / df_nrw["wohnheimplaetze_2025"]
)


In [7]:
df_nrw[[
    "hochschulort",
    "wohnheimplaetze_2025",
    "studierende_ws_2024_2025",
    "wohnheimplatzrelation"
]].sort_values("wohnheimplatzrelation").head(10)


,hochschulort,wohnheimplaetze_2025,studierende_ws_2024_2025,wohnheimplatzrelation
121,Bonn,4053.0,34744.0,8.572415
120,Bochum,5594.0,49371.0,8.825706
146,Münster,5997.0,54107.0,9.022345
118,Bielefeld,3193.0,31399.0,9.833699
147,Paderborn,1843.0,18135.0,9.839935
117,Aachen,5460.0,56493.0,10.346703
125,Düsseldorf,3597.0,40820.0,11.348346
124,Duisburg,950.0,13015.0,13.700000
137,Köln,5487.0,75655.0,13.788044
149,Siegen,1043.0,14439.0,13.843720


In [8]:
out_path_nrw = Path("../../data/processed/student_housing_nrw_scrub.csv")
df_nrw.to_csv(out_path_nrw, index=False)
out_path_nrw


WindowsPath('../../data/processed/student_housing_nrw_scrub.csv')

In [9]:
out_path_nrw.exists()


True

In [10]:
from pathlib import Path

out_path = Path("../../data/processed/student_housing_nrw_clean.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_nrw_clean.to_csv(out_path, index=False)
out_path.exists(), out_path.resolve()


NameError: name 'df_nrw_clean' is not defined

In [11]:
# Zeige alle DataFrames, die im Notebook existieren
import pandas as pd

[name for name, val in globals().items() if isinstance(val, pd.DataFrame)]


['___', 'df_de', 'df_nrw', '_7']

In [12]:
df_nrw_clean = df_nrw.copy()
df_nrw_clean.shape


(17, 6)

In [13]:
df_nrw_clean.columns, df_nrw_clean.shape


(Index(['hochschulort', 'wohnheime_2025_anzahl', 'wohnheimplaetze_2025',
        'studierende_ws_2024_2025', 'studierende_je_wohnheimplatz_2025',
        'wohnheimplatzrelation'],
       dtype='object'),
 (17, 6))

In [14]:
from pathlib import Path

out_path = Path("../../data/processed/student_housing_nrw_clean.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_nrw_clean.to_csv(out_path, index=False)
out_path.exists(), out_path.resolve()


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/processed/student_housing_nrw_clean.csv'))

In [15]:
df_nrw_clean = df_nrw.copy()
df_nrw_clean.shape


(17, 6)

In [16]:
from pathlib import Path

out_path = Path("../../data/processed/student_housing_nrw_clean.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)

df_nrw_clean.to_csv(out_path, index=False)

out_path.exists(), out_path.resolve()


(True,
 WindowsPath('C:/Users/User/Desktop/DATA INFORMATION SCIENCE/DIS08/dis08-miete-nrw/data/processed/student_housing_nrw_clean.csv'))